# CoalBlend AI — Demo End-to-End (tanpa Claude Desktop)

Notebook ini memanggil langsung fungsi Python di balik 3 MCP server
(`mcp_servers/spreadsheet_server.py`, `blend_server.py`, `dashboard_server.py`) —
tool `@mcp.tool()` tetap bisa dipanggil sebagai fungsi Python biasa, jadi cocok untuk
dites/didebug di Jupyter sebelum dipakai lewat Claude Desktop.

Alur: **spreadsheet template → ingest → blend (3 skenario) → render dashboard HTML**

**Sebelum run:** install dependency dulu (sekali saja):
```bash
pip install -r ../mcp_servers/requirements.txt
pip install -r ../requirements.txt
```

In [ ]:
import sys
from pathlib import Path

# Sesuaikan kalau folder project dipindah dari lokasi ini.
PROJECT_ROOT = Path(r"C:\Users\Riza\Downloads\coalblend_mvp_v3\coalblend_mvp1")
sys.path.insert(0, str(PROJECT_ROOT / "mcp_servers"))

import spreadsheet_server as sp
import blend_server as bl
import dashboard_server as db

print("Modules loaded OK")

## 1. Ingest — baca spreadsheet template (coalblend-data)

Bisa ganti `workbook_path` ke file real Anda sendiri (asal nama sheet & kolom sama
persis dengan template: `opening_stock`, `coal_getting`, `demand_deal`, `plan_kapasitas`).

In [ ]:
ingested = sp.ingest_coal_data_from_workbook("fixtures/coalblend_input_template.xlsx")

print("status:", ingested["status"])
print("alerts:", ingested["alerts"])
ingested["data"]

## 2. Blend — solve 3 skenario (coalblend-blend)

In [ ]:
result = bl.run_blending(ingested["data"])

print("status:", result["status"])
for sc in result["scenarios"]:
    print(f"  {sc['id']:>2} {sc['nama']:<32} revenue={sc['revenue']:>14,.2f}  delta={sc['delta_vs_baseline']:>+12,.2f}")

In [ ]:
baseline = bl.get_baseline_info()
print("expected baseline revenue:", baseline["baseline_validasi_revenue"])
assert abs(result["scenarios"][0]["revenue"] - baseline["baseline_validasi_revenue"]) < 1, "revenue tidak cocok baseline tervalidasi!"
print("OK — Opsi A cocok dengan baseline tervalidasi tests/test_pipeline.py")

## 3. Dashboard — render HTML statis (coalblend-dashboard)

In [ ]:
rendered = db.render_review_html(ingested["data"], out_path="output/dashboard_notebook_demo.html")
print("status:", rendered["status"])
print("path:", rendered["path"])
rendered["scenarios_summary"]

In [ ]:
# Tampilkan langsung di notebook
from IPython.display import IFrame
IFrame(src=rendered["path"], width="100%", height=600)

## 4. (Opsional) Terbitkan Report Final untuk 1 opsi yang dipilih

In [ ]:
report = db.render_report_html(
    ingested["data"],
    scenario_id="A",
    validator="partogi.luhut",
    catatan="demo notebook end-to-end",
    out_path="output/report_notebook_demo.html",
)
print(report)